# 👑 FINAL　最終 Boss：勇者咖啡 2.0 兩份預測報告
**統計冒險之旅 2026**　｜　Day 3（10/05 一）🗻 預測之巔　｜　最終 Boss　｜　🏅 500 XP

📖 全課綜合；訓練資料：勇者咖啡每日營收、會員；正式挑戰資料：**只有特徵，沒有答案標籤**。

### 🎯 這一關你會學到
- 迴歸篇：用訓練集建模，預測 21 筆每日營收，輸出 `final_daily_predictions.csv`
- 分類篇：預測 400 位會員的回購機率，輸出 `final_member_predictions.csv`
- 切分 → 基線 → 交叉驗證 → 調參 → 交付 prediction CSV → 教師端離線評分

### 🧭 闖關方式
1. 先執行下方「🧰 魔法工具箱」。
2. 依序完成 F-1 到 F-7；Notebook 只檢查流程、檔案結構與訓練內驗證。
3. 下載兩份 prediction CSV 交給教師；挑戰標籤只保留在教師端。
4. 全部任務通過後，再將通關密語貼回[入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.2/)。

> 🔐 公開 Notebook 中不含挑戰集的營收或回購答案；這是訓練、驗證、正式測試分離的實作。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "FINAL"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["F-1", "F-2", "F-3", "F-4", "F-5", "F-6", "F-7"]
_XP_EACH = 71
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------

def _check_F_1(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "每日營收平均"), 3642.288, 1): return (False, "每日營收平均 = daily['營收'].mean()。")
    if not 約等於(抓變數(ns, "回購比例"), 0.48400, 0.001): return (False, "回購比例 = members['回購'].mean()。")
    if int(抓變數(ns, "缺值總數")) != 0: return (False, "兩份訓練資料沒有缺值。")
    if 抓變數(ns, "挑戰資料無標籤") is not True: return (False, "公開挑戰資料不得包含營收或回購標籤。")
    return (tuple(抓變數(ns, "挑戰週日期")) == ("2026-09-01", "2026-09-07"), "挑戰週是 2026-09-01 到 2026-09-07。")
任務定義("F-1", _check_F_1, 提示="標籤只能出現在訓練資料，公開挑戰資料只有特徵。")

def _check_F_2(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "Xd"), 列=531, 欄=17)
    if not ok: return (False, msg + "（星期、分店、天氣三欄 one-hot，不用 drop_first）")
    return (約等於(抓變數(ns, "基線RMSE"), 926.899, 5), "基線RMSE = sqrt(mean_squared_error(yd_test, 基線.predict(Xd_test)))。")
任務定義("F-2", _check_F_2, 提示="np.sqrt(mean_squared_error(...))。")

def _check_F_3(run):
    out, ns = run()
    cv = 抓變數(ns, "CV_RMSE們", dict)
    if set(cv.keys()) != {"線性", "Ridge", "隨機森林"}: return (False, "CV_RMSE們 要有線性、Ridge、隨機森林三個。")
    if any(v < 0 for v in cv.values()): return (False, "scoring 是負的 RMSE，前面要加負號。")
    pred = 抓變數(ns, "每日預測檔")
    ok, msg = 資料框像(pred, 列=21, 欄=3, 含欄位=["日期", "分店", "預測營收"])
    if not ok: return (False, msg)
    if "營收" in pred.columns: return (False, "預測檔不得包含真實營收標籤。")
    values = pred["預測營收"].astype(float)
    if not (values.notna().all() and (values > 0).all()): return (False, "預測營收必須全為正數且不能缺值。")
    total = float(抓變數(ns, "下週預估總營收"))
    return (50000 < total < 110000, "下週預估總營收 = 預測週.sum()。")
任務定義("F-3", _check_F_3, 提示="要輸出日期、分店、預測營收三欄，不可有真實營收。")

def _check_F_4(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "Xm"), 列=2000, 欄=12)
    if not ok: return (False, msg)
    return (約等於(抓變數(ns, "基線AUC"), 0.80745, 0.01), "基線AUC = roc_auc_score(ym_test, 基線C.predict_proba(Xm_test)[:, 1])。")
任務定義("F-4", _check_F_4, 提示="predict_proba(...)[:, 1]。")

def _check_F_5(run):
    out, ns = run()
    pred = 抓變數(ns, "會員預測檔")
    ok, msg = 資料框像(pred, 列=400, 欄=2, 含欄位=["會員編號", "回購機率"])
    if not ok: return (False, msg)
    if "回購" in pred.columns: return (False, "預測檔不得包含真實回購標籤。")
    prob = pred["回購機率"].astype(float)
    if not (prob.notna().all() and prob.between(0, 1).all()): return (False, "回購機率必須介於 0 與 1。")
    lst = list(抓變數(ns, "名單"))
    if len(lst) != 300: return (False, "名單要剛好 300 人（head(300)）。")
    return (all(str(x).startswith("M") for x in lst), "名單要是會員編號（M 開頭）。")
任務定義("F-5", _check_F_5, 提示="輸出全部 400 位會員的機率，名單取機率最高的 300 位。")

def _check_F_6(run):
    out, ns = run()
    if len(run.figs) < 2: return (False, "要有兩張圖（兩個座標軸）。")
    t = " ".join(f["title"] for f in run.figs)
    return ("驗證" in t and "重要" in t, "標題要分別包含「驗證」與「重要」。")
任務定義("F-6", _check_F_6, 提示="圖只使用訓練內的驗證切分，不會看到挑戰集標籤。")

def _check_F_7(run):
    out, ns = run()
    d = 抓變數(ns, "結論", dict)
    keys = ["下週預估總營收", "驗證集RMSE", "最佳迴歸模型", "營收最重要因素", "驗證集AUC", "最可能回購的特徵", "優惠券名單筆數", "每日預測檔", "會員預測檔", "建議"]
    for k in keys:
        if k not in d: return (False, f"結論 缺少「{k}」。")
    if int(d["優惠券名單筆數"]) != 300: return (False, "優惠券名單筆數應該是 300。")
    if not str(d["每日預測檔"]).endswith(".csv") or not str(d["會員預測檔"]).endswith(".csv"): return (False, "兩個交付檔名都要是 .csv。")
    return (isinstance(d["建議"], str) and len(d["建議"]) >= 15, "建議要是一句至少 15 個字的話。")
任務定義("F-7", _check_F_7, 提示="教師會在課後以私有標籤計算正式 RMSE 與 AUC。")

## 👑 最終 Boss：勇者咖啡 2.0——兩份 prediction CSV

你有兩份可學到答案的訓練資料，以及兩份只有特徵的正式挑戰資料：

1. `coffee_daily.csv` → 學習預測營收；`coffee_daily_challenge.csv` → 輸出 21 筆預測營收。
2. `coffee_members.csv` → 學習預測回購；`coffee_members_challenge.csv` → 輸出 400 筆回購機率。

正式分數由教師端離線計算：迴歸用 RMSE，分類用 ROC AUC。你看不到挑戰集答案，也不能用它們調參。

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, roc_auc_score

BASE = "https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0-rc.2/data/"
daily = pd.read_csv(BASE + "coffee_daily.csv")
members = pd.read_csv(BASE + "coffee_members.csv")
挑戰週 = pd.read_csv(BASE + "final/coffee_daily_challenge.csv")
挑戰會員 = pd.read_csv(BASE + "final/coffee_members_challenge.csv")
print("訓練：", daily.shape, members.shape, "| 挑戰：", 挑戰週.shape, 挑戰會員.shape)
挑戰週.head(3)

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

### 🎯 任務 F-1　載入、初探與標籤邊界

算出 `每日營收平均`、`回購比例`、`缺值總數`（兩份訓練資料的缺值加總），把挑戰週日期範圍存成 `挑戰週日期`，並檢查兩份挑戰資料不含目標欄位，存成 `挑戰資料無標籤`。

In [ ]:
# 🎯 任務 F-1　載入、初探與標籤邊界（請保留這一行）
每日營收平均 = daily["營收"].mean()
回購比例 = ???
缺值總數 = int(daily.isna().sum().sum() + members.isna().sum().sum())
挑戰週日期 = (挑戰週["日期"].min(), 挑戰週["日期"].max())
挑戰資料無標籤 = ("營收" not in 挑戰週.columns) and ("回購" not in 挑戰會員.columns)
print(round(每日營收平均, 1), round(回購比例, 3), 缺值總數, 挑戰週日期, 挑戰資料無標籤)

In [ ]:
檢查("F-1")   # ◀ 執行這一格，看看任務 F-1 有沒有過關

### 🎯 任務 F-2　迴歸篇：切分與基線

把 `daily` 轉成特徵 `Xd`（去掉 日期、營收，`星期、分店、天氣` 做 one-hot，不用 drop_first，`.astype(float)`）與目標 `yd`；`test_size=0.2, random_state=42` 切分；用線性迴歸當**基線**，算 `基線RMSE`（測試集）。

In [ ]:
# 🎯 任務 F-2　迴歸篇：切分與基線（請保留這一行）
Xd = pd.get_dummies(daily.drop(columns=["日期", "營收"]), columns=["星期", "分店", "天氣"]).astype(float)
yd = daily["營收"]
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size=0.2, random_state=42)
基線 = LinearRegression().fit(Xd_train, yd_train)
基線RMSE = ???
print(Xd.shape, round(基線RMSE, 1))

In [ ]:
檢查("F-2")   # ◀ 執行這一格，看看任務 F-2 有沒有過關

### 🎯 任務 F-3　迴歸篇：交叉驗證選模型、輸出挑戰週預測

用訓練切分比較線性迴歸、Ridge 與隨機森林的 5-fold CV RMSE；保留訓練內的 `驗證集RMSE`，再用全部訓練資料拟合最佳模型。對 21 筆無標籤挑戰資料預測，產生只含 `日期、分店、預測營收` 的 `每日預測檔`，並另存為 `final_daily_predictions.csv`。

In [ ]:
# 🎯 任務 F-3　迴歸篇：選模型與輸出預測 CSV（請保留這一行）
候選 = {"線性": LinearRegression(),
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10)),
        "隨機森林": RandomForestRegressor(300, random_state=42)}
CV_RMSE們 = {name: -cross_val_score(m, Xd_train, yd_train, cv=5,
                                    scoring="neg_root_mean_squared_error").mean()
            for name, m in 候選.items()}
最佳迴歸模型 = min(CV_RMSE們, key=CV_RMSE們.get)
驗證模型R = clone(候選[最佳迴歸模型]).fit(Xd_train, yd_train)
驗證集RMSE = np.sqrt(mean_squared_error(yd_test, 驗證模型R.predict(Xd_test)))
模型R = clone(候選[最佳迴歸模型]).fit(Xd, yd)
Xd_challenge = pd.get_dummies(挑戰週.drop(columns=["日期"]),
                              columns=["星期", "分店", "天氣"]).astype(float).reindex(columns=Xd.columns, fill_value=0)
預測週 = 模型R.predict(Xd_challenge)
下週預估總營收 = ???
每日預測檔 = 挑戰週[["日期", "分店"]].copy()
每日預測檔["預測營收"] = 預測週
每日預測檔.to_csv("final_daily_predictions.csv", index=False)
print({k: round(v, 1) for k, v in CV_RMSE們.items()}, 最佳迴歸模型, "| 驗證 RMSE", round(驗證集RMSE, 1))
print(每日預測檔.head())

In [ ]:
檢查("F-3")   # ◀ 執行這一格，看看任務 F-3 有沒有過關

### 🎯 任務 F-4　分類篇：切分與基線

把 `members` 轉成 `Xm`（去掉 會員編號、回購 後 one-hot，`drop_first=True`、`.astype(float)`）與 `ym`；`test_size=0.25, random_state=42, stratify=ym` 切分；基線用 Pipeline（StandardScaler → LogisticRegression(max_iter=1000)），算測試集 `基線AUC`。

In [ ]:
# 🎯 任務 F-4　分類篇：切分與基線（請保留這一行）
Xm = pd.get_dummies(members.drop(columns=["會員編號", "回購"]), drop_first=True).astype(float)
ym = members["回購"]
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.25, random_state=42, stratify=ym)
基線C = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xm_train, ym_train)
基線AUC = ???
print(Xm.shape, round(基線AUC, 3))

In [ ]:
檢查("F-4")   # ◀ 執行這一格，看看任務 F-4 有沒有過關

### 🎯 任務 F-5　分類篇：調參、輸出回購機率與優惠券名單

用 `GridSearchCV` 調整隨機森林，先在訓練內切分保留 `驗證集AUC`，再使用全部訓練資料拟合。對 400 位挑戰會員預測回購機率，產生只含 `會員編號、回購機率` 的 `會員預測檔`，另存為 `final_member_predictions.csv`；再取機率最高的 300 位會員編號存到 `名單`。

In [ ]:
# 🎯 任務 F-5　分類篇：調參與輸出預測 CSV（請保留這一行）
gs = GridSearchCV(RandomForestClassifier(200, random_state=42),
                  {"max_depth": [5, 8, None], "min_samples_leaf": [1, 5, 20]},
                  cv=5, scoring="roc_auc").fit(Xm_train, ym_train)
驗證集AUC = roc_auc_score(ym_test, gs.best_estimator_.predict_proba(Xm_test)[:, 1])
模型C = clone(gs.best_estimator_).fit(Xm, ym)
Xm_challenge = pd.get_dummies(挑戰會員.drop(columns=["會員編號"]),
                              drop_first=True).astype(float).reindex(columns=Xm.columns, fill_value=0)
挑戰機率 = 模型C.predict_proba(Xm_challenge)[:, 1]
會員預測檔 = 挑戰會員[["會員編號"]].copy()
會員預測檔["回購機率"] = 挑戰機率
會員預測檔.to_csv("final_member_predictions.csv", index=False)
排名 = 會員預測檔.sort_values("回購機率", ascending=False)
名單 = 排名["會員編號"].head(???).tolist()
print(gs.best_params_, "| 驗證集 AUC", round(驗證集AUC, 3), "| 名單", len(名單), "人")
print(會員預測檔.head())

In [ ]:
檢查("F-5")   # ◀ 執行這一格，看看任務 F-5 有沒有過關

### 🎯 任務 F-6　兩張訓練內診斷圖

畫 (1) 迴歸的「驗證集：實際 vs 預測」散佈圖，加 45 度線；(2) 分類模型的特徵重要性橫條圖。圖表只用訓練內的驗證切分，不使用正式挑戰標籤。

In [ ]:
# 🎯 任務 F-6　兩張圖（請保留這一行）
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
驗證預測 = 驗證模型R.predict(Xd_test)
ax[0].scatter(yd_test, 驗證預測)
lim = [yd_test.min() * .9, yd_test.max() * 1.1]
ax[0].plot(lim, lim, "--", color="grey")
ax[0].set_xlabel("驗證集實際營收"); ax[0].set_ylabel("驗證集預測營收"); ax[0].set_title(???)
重要性 = pd.Series(模型C.feature_importances_, index=Xm.columns).sort_values()
重要性.plot(kind="barh", ax=ax[1]); ax[1].set_title(???)
plt.tight_layout(); plt.savefig("final_diagnostics.png"); plt.show()

In [ ]:
檢查("F-6")   # ◀ 執行這一格，看看任務 F-6 有沒有過關

### 🎯 任務 F-7　給老闆的兩份結論與交付清單

把結果填進字典 `結論`：`下週預估總營收、驗證集RMSE、最佳迴歸模型、營收最重要因素、驗證集AUC、最可能回購的特徵、優惠券名單筆數、每日預測檔、會員預測檔、建議`。

> 報告中的 RMSE 與 AUC 是訓練內驗證證據，不是正式挑戰成績；挑戰成績由教師端收件後另行計算。

In [ ]:
# 🎯 任務 F-7　給老闆的結論與交付清單（請保留這一行）
if 最佳迴歸模型 == "隨機森林":
    營收最重要因素 = pd.Series(模型R.feature_importances_, index=Xd.columns).idxmax()
else:
    係數R = 模型R[-1].coef_ if hasattr(模型R, "steps") else 模型R.coef_
    營收最重要因素 = pd.Series(np.abs(係數R), index=Xd.columns).idxmax()
結論 = {
    "下週預估總營收": round(下週預估總營收),
    "驗證集RMSE": round(驗證集RMSE),
    "最佳迴歸模型": 最佳迴歸模型,
    "營收最重要因素": 營收最重要因素,
    "驗證集AUC": round(驗證集AUC, 3),
    "最可能回購的特徵": 重要性.idxmax(),
    "優惠券名單筆數": len(名單),
    "每日預測檔": "final_daily_predictions.csv",
    "會員預測檔": "final_member_predictions.csv",
    "建議": ???,
}
for k, v in 結論.items():
    print(f"{k}：{v}")

In [ ]:
檢查("F-7")   # ◀ 執行這一格，看看任務 F-7 有沒有過關

## 🎤 成果分享（3 分鐘）

用訓練內驗證證據與兩份 prediction CSV 向「老闆」報告：下週該備多少貨？優惠券該優先寄給誰？哪一個診斷是你最有把握的？正式挑戰分數尚未回傳前，不要把驗證分數說成正式成績。

## 🌟 進階挑戰（不計分）
1. 把 L09 的分群結果當成新特徵，訓練內的 AUC 有沒有提高？
2. 用 bootstrap 給「下週預估總營收」一個 95% 區間。
3. 把 `上週同日營收` 拿掉再做交叉驗證，比較 RMSE 差異。

---
## 🔑 通關密語
　三峰登頂！你已經完成從資料到決策的完整旅程。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**你已經登上預測之巔！請回入口網頁查看學習完成紀錄。**

> 此紀錄不是補習班正式結業證書；請另將 FINAL 預測檔交由教師離線驗收。

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.2/